# 建筑 KPI 异常数据集构建（`tool` 版）

以仓库根目录下的 [`fwwb01_realistic_with_simulated_kpis.csv`](../fwwb01_realistic_with_simulated_kpis.csv) 为输入，在**带随机噪声的观测列**上注入多类可解释异常；**期望值**来自 `*_nominal` 并写入单独基线表，供规则检测模块做「相对 + 绝对」双阈对比。

## 与「分层诊断 + 双阈」检测方案的对齐关系

**症状层（并行，无顺序锁）**：水、总电、空调电、供/回水温度偏离各自独立判据；注入时尽量让每一类至少覆盖若干小时，以便统计召回。

**根因层（仅当空调电异常）**：`building_anomaly_detection.py` 中 `hvac_diagnosis` 的 A/B/C 与下列物理叙事对应——

- **A（主机/冷却塔/供水传感器）**：制冷工况下 `hac_energy` 偏高且 **供水**相对期望偏离 >1.5℃。
- **B（水力失调 / 小温差）**：制冷、`hac` 偏高、供水接近期望，但 **瞬时温差** \((return-supply) < 3\)℃ 且与期望温差模式不符。
- **C（非预期开机 / 排程）**：**非营业时段**（周末或深夜）`hac` 绝对值高而期望低，温差仍「正常」区间。

## 异常类型矩阵（1–10 为原版场景，11–13 为扩展）

| ID | anomaly_name | 业态 | 时序/季节 | 主要异常通道 | 易触发症状 | 典型根因标签 |
|---:|---|---|---|---|---|---|
| 1 | office_weekend_schedule_failure | 办公 | 周六全天 | 空调电+供回水（制冷设定） | flag_hvac, flag_supply, flag_return | C |
| 2 | office_masking_base_down_hvac_up | 办公 | 夏季周五傍晚 | 总电+空调（基载被压低） | flag_hvac（总电可能被掩蔽） | B/A 视温差 |
| 3 | office_low_delta_t_bypass | 办公 | 夏季高温连续小时 | 空调电+回水抬高 | flag_hvac, flag_return | B |
| 4 | teaching_water_leak | 教学 | 不限 | 水耗 | flag_water | — |
| 5 | teaching_open_window_cooling_waste | 教学 | 春夏/秋初、高温小时 | 空调+回水 | flag_hvac, flag_return | B/其他 |
| 6 | teaching_chiller_degradation_heatwave | 教学 | 酷暑午后 | 空调+供水升高 | flag_hvac, flag_supply | A |
| 7 | library_high_dew_dehumidification | 图书 | 高露点段 | 空调+供水 | flag_hvac, flag_supply | A/其他 |
| 8 | library_night_lights_left_on | 图书 | 闭馆夜间 | **仅总电**（空调不大动） | flag_electricity | — |
| 9 | library_transition_valve_leak | 图书 | 过渡季长段 | 空调+供水 | flag_hvac, flag_supply | A/其他 |
| 10 | library_supply_sensor_drift | 图书 | 不限 | **仅供水读数** | flag_supply | —（无 hac 异常时无根因工单） |
| 11 | office_non_hvac_electricity_surge | 办公 | **工作日白天** | **仅总电** | flag_electricity | — |
| 12 | office_summer_water_leak | 办公 | **夏季高温** | 水耗 | flag_water | — |
| 13 | office_return_temp_sensor_bias | 办公 | 制冷小时 | **仅回水读数**（能耗不变） | flag_return | — |

说明：`water_l` 单位为 **L**；水规则绝对项 1.0 m³ = **1000 L**，与 `DetectionThresholds.water_abs` 一致。

## 实现说明

注入逻辑在仓库根目录的 [`kpi_anomaly_inject.py`](../kpi_anomaly_inject.py) 中实现（`run_kpi_injection`）。本 notebook 只负责路径解析与调用。默认输出到 **`tool/output/`**，避免覆盖根目录下原有 `fwwb01_*anomalies*.csv`。

In [1]:
from pathlib import Path
import sys


def resolve_repo_root() -> Path:
    c = Path.cwd().resolve()
    for p in (c, c.parent):
        if (p / "kpi_anomaly_inject.py").is_file() and (
            p / "fwwb01_realistic_with_simulated_kpis.csv"
        ).is_file():
            return p
    raise FileNotFoundError(
        "未找到 kpi_anomaly_inject.py 与 fwwb01_realistic_with_simulated_kpis.csv；"
        "请将 Jupyter 工作目录设为仓库根目录 e:/yp/data 或其子目录 tool"
    )


ROOT = resolve_repo_root()
sys.path.insert(0, str(ROOT))

from kpi_anomaly_inject import run_kpi_injection

INPUT_CSV = ROOT / "fwwb01_realistic_with_simulated_kpis.csv"
OUT_DIR = ROOT / "tool" / "output"
OUT_DIR.mkdir(parents=True, exist_ok=True)

run_kpi_injection(
    INPUT_CSV,
    OUT_DIR / "kpi_with_anomalies.csv",
    OUT_DIR / "kpi_baseline_for_detection.csv",
    OUT_DIR / "kpi_anomaly_labels.csv",
    rng_seed=20260221,
    include_extended_types=True,
    verbose=True,
)

saved: E:\yp\data\tool\output\kpi_with_anomalies.csv E:\yp\data\tool\output\kpi_baseline_for_detection.csv E:\yp\data\tool\output\kpi_anomaly_labels.csv
label rows: 2972 rows in df: 26316
             timestamp       building_id  anomaly_type_id  \
0  2016-09-02 20:00:00  Fox_office_Alice                1   
1  2016-09-02 21:00:00  Fox_office_Alice                1   
2  2016-09-02 22:00:00  Fox_office_Alice                1   
3  2016-09-02 23:00:00  Fox_office_Alice                1   
4  2016-09-03 00:00:00  Fox_office_Alice                1   
5  2016-09-03 01:00:00  Fox_office_Alice                1   
6  2016-09-03 02:00:00  Fox_office_Alice                1   
7  2016-09-03 03:00:00  Fox_office_Alice                1   
8  2016-09-03 04:00:00  Fox_office_Alice                1   
9  2016-09-03 05:00:00  Fox_office_Alice                1   
10 2016-09-03 06:00:00  Fox_office_Alice                1   
11 2016-09-03 07:00:00  Fox_office_Alice                1   

                  

(                 timestamp          building_id primaryspaceusage      sqm  \
 0      2016-01-01 00:00:00  Fox_education_Louie         Education  1664.52   
 1      2016-01-01 01:00:00  Fox_education_Louie         Education  1664.52   
 2      2016-01-01 02:00:00  Fox_education_Louie         Education  1664.52   
 3      2016-01-01 03:00:00  Fox_education_Louie         Education  1664.52   
 4      2016-01-01 04:00:00  Fox_education_Louie         Education  1664.52   
 ...                    ...                  ...               ...      ...   
 26311  2016-12-31 18:00:00     Fox_public_Henry   Public services  1251.30   
 26312  2016-12-31 20:00:00     Fox_public_Henry   Public services  1251.30   
 26313  2016-12-31 21:00:00     Fox_public_Henry   Public services  1251.30   
 26314  2016-12-31 22:00:00     Fox_public_Henry   Public services  1251.30   
 26315  2016-12-31 23:00:00     Fox_public_Henry   Public services  1251.30   
 
        airTemperature  dewTemperature  humidity  